## Feature Engineering

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}"
    f"@{os.getenv('MYSQL_HOST')}/{os.getenv('MYSQL_DATABASE')}"
)

In [3]:
query = """
SELECT d.record_date, d.train_no, d.station_code, d.station_no,
       d.delay_minutes, d.day_of_week, d.month, d.is_monsoon,
       d.is_extreme_delay, t.coverage_tier, s.station_zone
FROM delays d
JOIN trains t ON d.train_no = t.train_no
JOIN stations s ON d.station_code = s.station_code
"""
df = pd.read_sql(query, engine)
df["record_date"] = pd.to_datetime(df["record_date"])
df.shape

(262256, 11)

In [4]:
station_to_region = {
    "CSMT": "Mumbai", "CSTM": "Mumbai", "DR": "Mumbai", "TNA": "Mumbai",
    "PNVL": "Mumbai", "BDTS": "Mumbai", "LTT": "Mumbai",
    "ROHA": "Mumbai", "VEER": "Mumbai",
    "SNGD": "Ratnagiri", "CHI":  "Ratnagiri", "KHED": "Ratnagiri", "RN":   "Ratnagiri", "ADL":  "Ratnagiri", 
    "VBW":  "Kankavli", "KKW": "Kankavali", "NAN": "Kankavali",
     "SNDD": "Sawantwadi", "KUDL": "Sawantwadi","SWV": "Sawantwadi", "ZARP": "Sawantwadi",
    "MAO": "Madgaon", "KRMI": "Madgaon", "THVM": "Madgaon", "PERN": "Madgaon",
    "MAQ": "Mangalore", "MAJN": "Mangalore", "SL": "Mangalore", "MULK": "Mangalore", "CANO": "Mangalore", "UD":   "Mangalore",
}
weather_df = pd.read_csv("../data/raw/weather/konkan_rainfall_meteostat.csv")
weather_df["date"] = pd.to_datetime(weather_df["date"])

df["region"] = df["station_code"].map(station_to_region)

In [5]:
unmapped = df.loc[df["region"].isna(), "station_code"].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} station_code values have no region mapping:")
    print(sorted(unmapped))

df = df.merge(
    weather_df[["date", "region", "rainfall_mm"]],
    left_on=["record_date", "region"], right_on=["date", "region"], how="left",
)

['ABR', 'ACRN', 'ADI', 'ADVI', 'AGC', 'AKV', 'ALLP', 'AMPA', 'ANKL', 'ANND', 'ANO', 'APTA', 'ARA', 'AT', 'AVRD', 'AWY', 'BAU', 'BAW', 'BDJ', 'BGNR', 'BINA', 'BIRD', 'BKJ', 'BKN', 'BKNG', 'BL', 'BNC', 'BNTL', 'BOR', 'BPL', 'BRC', 'BSB', 'BSL', 'BSR', 'BTJL', 'BVI', 'BXR', 'BYNR', 'CAN', 'CBE', 'CDG', 'CHV', 'CLT', 'CNGR', 'CNO', 'CNPA', 'CPT', 'CUR', 'CVP', 'DDU', 'DG', 'DIVA', 'DNR', 'ED', 'ERN', 'ERS', 'ET', 'FA', 'FDB', 'FK', 'GDL', 'GIMB', 'GNO', 'GOK', 'GWL', 'GYN', 'HAA', 'HAD', 'HAS', 'HLN', 'HNA', 'HSR', 'IGP', 'JBP', 'JITE', 'JL', 'JND', 'JU', 'KAWR', 'KBPR', 'KFD', 'KGI', 'KGQ', 'KIGL', 'KLMG', 'KNW', 'KOL', 'KOTA', 'KPQ', 'KPY', 'KRNR', 'KRPN', 'KRR', 'KSD', 'KT', 'KTE', 'KTU', 'KTYM', 'KUDA', 'KYJ', 'KYN', 'KZE', 'LUNI', 'MAN', 'MANK', 'MDU', 'MJ', 'MKP', 'MMR', 'MNI', 'MRA', 'MRDW', 'MRJN', 'MSH', 'MTD', 'MTJ', 'MVLK', 'MYA', 'MYS', 'NDB', 'NDLS', 'NGO', 'NIV', 'NJT', 'NK', 'NLE', 'NMGA', 'NOK', 'NUD', 'NVS', 'NZM', 'PAY', 'PAZ', 'PDD', 'PEN', 'PGI', 'PGT', 'PMY', 'PNP', 'P

In [6]:
def rain_category(mm):
    if pd.isna(mm): return "Unknown"
    elif mm < 2.5: return "No rain"
    elif mm < 15.6: return "Light"
    elif mm < 64.5: return "Moderate"
    elif mm < 115.6: return "Heavy"
    else: return "Very heavy"

df["rain_category"] = df["rainfall_mm"].apply(rain_category)
df.drop(columns=["date"], inplace=True)

In [7]:
for col in ["station_code", "train_no", "day_of_week", "rain_category", "station_zone"]:
    print(col, df[col].nunique())

station_code 218
train_no 46
day_of_week 7
rain_category 6
station_zone 11


In [8]:
categorical_cols = ["station_code", "train_no", "day_of_week", "rain_category", "station_zone"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape

(262256, 292)

### Train/Test Split + Baseline

In [10]:
df_encoded = df_encoded.sort_values("record_date")

split_date = df_encoded["record_date"].quantile(0.85, interpolation="nearest")
print(f"Split date: {split_date}")

train_df = df_encoded[df_encoded["record_date"] < split_date]
test_df = df_encoded[df_encoded["record_date"] >= split_date]

print(f"Train: {train_df.shape[0]} rows ({train_df['record_date'].min()} to {train_df['record_date'].max()})")
print(f"Test:  {test_df.shape[0]} rows ({test_df['record_date'].min()} to {test_df['record_date'].max()})")

Split date: 2025-12-16 00:00:00
Train: 222450 rows (2025-02-08 00:00:00 to 2025-12-15 00:00:00)
Test:  39806 rows (2025-12-16 00:00:00 to 2026-02-07 00:00:00)


In [11]:
drop_cols = ["record_date", "delay_minutes", "region", "rainfall_mm", "is_extreme_delay", "coverage_tier"]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df["delay_minutes"]
X_test = test_df.drop(columns=drop_cols)
y_test = test_df["delay_minutes"]

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Baseline - MAE: {mae:.2f} min, RMSE: {rmse:.2f} min, R2: {r2:.3f}")

Baseline - MAE: 34.59 min, RMSE: 60.55 min, R2: 0.051


In [14]:
dumb_preds = np.full_like(y_test, y_train.mean(), dtype=float)
dumb_mae = mean_absolute_error(y_test, dumb_preds)
print(f"Predict-the-mean MAE: {dumb_mae:.2f} min")
print(f"Linear regression improvement: {dumb_mae - mae:.2f} min")

Predict-the-mean MAE: 39.06 min
Linear regression improvement: 4.47 min
